In [77]:
import pandas as pd

from google.colab import drive

drive.mount ('/content/drive/')

origem = '/content/drive/My Drive/datasus/cnvtocsv/'

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [78]:
arquivo_1 = "dt_final_2.csv"
arquivo_2 = "IDADE.csv"
arquivo_3 = "CID10_20.csv"
arquivo_4 = "SEXO.csv"
arquivo_5 = "ESCAGR1.csv"
arquivo_6 = "CBO2002.csv"
arquivo_7 = "LOCOCOR.csv"

In [79]:
dados = pd.read_csv(origem + arquivo_1, low_memory=False, sep= ',', encoding='latin1')
idadet = pd.read_csv(origem + arquivo_2, low_memory=False, sep= ';', encoding='latin1')
causat = pd.read_csv(origem + arquivo_3, low_memory=False, sep= ';', encoding='latin1')
sexot = pd.read_csv(origem + arquivo_4, low_memory=False, sep= ';', encoding='latin1')
esct = pd.read_csv(origem + arquivo_5, low_memory=False, sep= ';', encoding='latin1')
ocupt = pd.read_csv(origem + arquivo_6, low_memory=False, sep= ';', encoding='latin1')
munt = pd.read_csv(origem + arquivo_7, low_memory=False, sep= ';', encoding='latin1')

In [80]:
caminho_csv = '/content/drive/MyDrive/datasus/cnvtocsv/MUNICBR.csv'

munit = pd.read_csv(
    caminho_csv,
    sep=',',
    encoding='utf-8-sig',
    dtype=str,
    quotechar='"'
)

In [81]:
dados_temp = pd.read_csv(origem + arquivo_1, usecols=['IDADE'], low_memory=False)
dados['IDADE'] = dados_temp['IDADE']

idadet['codigo'] = idadet['codigo'].astype(str).str.strip()

dados['IDADE'] = pd.to_numeric(dados['IDADE'], errors='coerce').fillna(-1).astype(int).astype(str)

mapeamento_idade = dict(zip(idadet['codigo'], idadet['descricao']))

dados['IDADE'] = dados['IDADE'].map(mapeamento_idade)

dados['IDADE'] = dados['IDADE'].astype(str).str.extract(r'(\d+)')

In [82]:
causat['codigo'] = causat['codigo'].astype(str).str.strip()

dados['CAUSABAS'] = dados['CAUSABAS'].astype(str).str.strip()

mapeamento_causa = dict(zip(causat['codigo'], causat['descricao']))

dados['CAUSABAS'] = dados['CAUSABAS'].map(mapeamento_causa).fillna(dados['CAUSABAS'])

def corrigir_encoding(texto):
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin1').decode('utf-8')
    except Exception:
        return str(texto)

dados['CAUSABAS'] = dados['CAUSABAS'].apply(corrigir_encoding)

In [83]:
dados_temp_sexo = pd.read_csv(origem + arquivo_1, usecols=['SEXO'], low_memory=False)
dados['SEXO'] = dados_temp_sexo['SEXO']

sexot_temp = sexot.copy()
sexot_temp['codigo_limpo'] = sexot_temp['codigo'].astype(str).str.extract(r'(\d+)')

mapeamento_extenso = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'}
sexot_temp['descricao_extenso'] = sexot_temp['descricao'].map(mapeamento_extenso).fillna(sexot_temp['descricao'])

mapeamento_sexo_final = {}
for _, row in sexot_temp.iterrows():
    cod = row['codigo_limpo']
    desc = row['descricao_extenso']
    if pd.notna(cod):
        mapeamento_sexo_final[str(cod)] = desc

dados['SEXO'] = pd.to_numeric(dados['SEXO'], errors='coerce').fillna(-1).astype(int).astype(str)

dados['SEXO'] = dados['SEXO'].map(mapeamento_sexo_final).fillna('Ignorado')

In [84]:
dados_temp_esc = pd.read_csv(origem + arquivo_1, usecols=['ESC2010'], low_memory=False)
dados['ESC2010'] = dados_temp_esc['ESC2010']

esct_temp = esct.copy()
esct_temp['codigo'] = esct_temp['codigo'].astype(str).str.strip().str.zfill(2)

mapeamento_esc = dict(zip(esct_temp['codigo'], esct_temp['descricao']))

dados['ESC2010'] = pd.to_numeric(dados['ESC2010'], errors='coerce').fillna(9).astype(int).astype(str).str.zfill(2)

dados['ESC2010'] = dados['ESC2010'].map(mapeamento_esc).fillna('Ignorado')

def corrigir_encoding_esc(texto):
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin1').decode('utf-8')
    except Exception:
        return str(texto)

dados['ESC2010'] = dados['ESC2010'].apply(corrigir_encoding_esc)

In [85]:
dados_temp_loc = pd.read_csv(origem + arquivo_1, usecols=['LOCOCOR'], low_memory=False)
dados['LOCOCOR'] = dados_temp_loc['LOCOCOR']

munt_temp = munt.copy()
munt_temp['codigo'] = munt_temp['codigo'].astype(str).str.strip()

mapeamento_loc = dict(zip(munt_temp['codigo'], munt_temp['descricao']))

dados['LOCOCOR'] = pd.to_numeric(dados['LOCOCOR'], errors='coerce').fillna(9).astype(int).astype(str)

dados['LOCOCOR'] = dados['LOCOCOR'].map(mapeamento_loc).fillna('Ignorado')

def corrigir_encoding_loc(texto):
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin1').decode('utf-8')
    except Exception:
        return str(texto)

dados['LOCOCOR'] = dados['LOCOCOR'].apply(corrigir_encoding_loc)

In [86]:
dados_temp_hora = pd.read_csv(origem + arquivo_1, usecols=['HORAOBITO'], low_memory=False)
dados['HORAOBITO'] = dados_temp_hora['HORAOBITO']

def formatar_hora(valor):
    if pd.isna(valor):
        return 'Ignorado'

    val_str = str(valor).strip()

    if '.' in val_str:
        val_str = val_str.split('.')[0]

    val_str = ''.join(filter(str.isdigit, val_str))

    if not val_str:
        return 'Ignorado'

    if len(val_str) == 4:
        return f"{val_str[:2]}:{val_str[2:]}"

    elif len(val_str) == 3:
        val_str = '0' + val_str
        return f"{val_str[:2]}:{val_str[2:]}"

    return val_str

dados['HORAOBITO'] = dados['HORAOBITO'].apply(formatar_hora).fillna('Ignorado')

In [87]:
dados_temp_data = pd.read_csv(origem + arquivo_1, usecols=['DTOBITO'], low_memory=False)
dados['DTOBITO'] = dados_temp_data['DTOBITO']

def formatar_data(valor):
    if pd.isna(valor):
        return 'Ignorado'

    val_str = str(valor).strip()
    if '.' in val_str:
        val_str = val_str.split('.')[0]

    val_str = ''.join(filter(str.isdigit, val_str))

    if not val_str:
        return 'Ignorado'

    if len(val_str) == 7:
        val_str = '0' + val_str

    if len(val_str) == 8:
        dia = val_str[:2]
        mes = val_str[2:4]
        ano = val_str[4:]
        return f"{dia}/{mes}/{ano}"

    return 'Ignorado'

dados['DTOBITO'] = dados['DTOBITO'].apply(formatar_data)

In [88]:
dados_temp_ocup = pd.read_csv(origem + arquivo_1, usecols=['OCUP'], low_memory=False)
dados['OCUP'] = dados_temp_ocup['OCUP']

ocupt_temp = ocupt.copy()

def extrair_codigo_limpo(val):
    val_str = str(val).split(',')[0].strip()
    return ''.join(filter(str.isdigit, val_str))

ocupt_temp['codigo_limpo'] = ocupt_temp['codigo'].apply(extrair_codigo_limpo)

mapeamento_ocup = dict(zip(ocupt_temp['codigo_limpo'], ocupt_temp['descricao']))

dados['OCUP'] = dados['OCUP'].astype(str).apply(lambda x: ''.join(filter(str.isdigit, x.split('.')[0])))

dados['OCUP'] = dados['OCUP'].map(mapeamento_ocup).fillna('Ignorado')

def corrigir_encoding_ocup(texto):
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin1').decode('utf-8')
    except Exception:
        return str(texto)

dados['OCUP'] = dados['OCUP'].apply(corrigir_encoding_ocup)


In [89]:
dados_temp_mun = pd.read_csv(origem + arquivo_1, usecols=['CODMUNOCOR'], low_memory=False)
dados['CODMUNOCOR'] = dados_temp_mun['CODMUNOCOR']

munit_temp = munit.copy()
munit_temp['codigo'] = munit_temp['codigo'].astype(str).str.strip()
dados['CODMUNOCOR'] = dados['CODMUNOCOR'].astype(str).str.strip()

mapeamento_mun = dict(zip(munit_temp['codigo'], munit_temp['descricao']))

dados['CODMUNOCOR'] = dados['CODMUNOCOR'].map(mapeamento_mun).fillna('Ignorado')

def corrigir_encoding_mun(texto):
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin1').decode('utf-8')
    except Exception:
        return str(texto)

dados['CODMUNOCOR'] = dados['CODMUNOCOR'].apply(corrigir_encoding_mun)

In [90]:
dados.head()

,CAUSABAS,DTOBITO,HORAOBITO,IDADE,SEXO,ESC2010,OCUP,LOCOCOR,CODMUNOCOR
0,Contato com serpentes e lagartos venenosos - f...,03/06/2020,12:38,68,Feminino,Fundamental I Incompleto,Caseiro (agricultura),Hospital,Fortaleza
1,Agressão por meio de disparo de outra arma de ...,29/05/2020,19:49,27,Masculino,Fundamental I Completo,Vigia,Hospital,Guaiúba
2,Agressão por meio de disparo de outra arma de ...,02/06/2020,20:00,24,Masculino,Fundamental I Completo,Nao informada,Domicílio,Fortaleza
3,Agressão por meio de disparo de outra arma de ...,17/05/2020,01:32,35,Masculino,Fundamental II Incompleto,Estudante,Outros,Maracanaú
4,Agressão por meio de disparo de outra arma de ...,29/05/2020,19:40,37,Masculino,Fundamental I Completo,Servente de obras,Hospital,Guaiúba


In [91]:
caminho_salvar = origem + 'dt_2025.csv'
dados.to_csv(caminho_salvar, index=False, sep=';', encoding='utf-8-sig')